# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal de clasificación con **Keras 3** sobre Fashion-MNIST. Ahora resolverás un problema más desafiante con el mismo flujo de trabajo: **CIFAR-10**.

CIFAR-10 contiene **60 000 imágenes de 32×32 píxeles** divididas en **10 clases**:

| Label | Clase |
|:---:|:---|
| 0 | airplane |
| 1 | automobile |
| 2 | bird |
| 3 | cat |
| 4 | deer |
| 5 | dog |
| 6 | frog |
| 7 | horse |
| 8 | ship |
| 9 | truck |

Para que el modelo fully connected sea manejable y mantengamos la continuidad con la clase, convertiremos las imágenes a **escala de grises** ($32 \times 32 \times 1$).

---


---
## 0. Librerías


In [ ]:
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: CIFAR-10

CIFAR-10 contiene 50,000 imágenes de entrenamiento y 10,000 de prueba. Las imágenes originales son de **32×32×3** y pertenecen a 10 categorías:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`.


In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("X_train original:", X_train.shape)
print("X_test original :", X_test.shape)
print("y_train         :", y_train.shape)


## 1.1 RGB → escala de grises

Usaremos la transformación:

$$Gray = 0.299R + 0.587G + 0.114B$$

Esta parte está resuelta porque el objetivo del ejercicio es practicar Keras.


In [ ]:
X_train = (
    0.299 * X_train[..., 0] +
    0.587 * X_train[..., 1] +
    0.114 * X_train[..., 2]
)

X_test = (
    0.299 * X_test[..., 0] +
    0.587 * X_test[..., 1] +
    0.114 * X_test[..., 2]
)

print("X_train grayscale:", X_train.shape)
print("X_test grayscale :", X_test.shape)


### Pregunta 1

Después de convertir las imágenes a escala de grises, cada imagen tiene tamaño `32×32`.

¿Cuántos valores tendrá cada imagen después de aplicar `Flatten()`?

**Respuesta:**

Cada imagen tendrá $32 \times 32 = 1,024$ valores, correspondientes a la vectorización unidimensional de la matriz de píxeles.


## 1.2 Normalización

Los valores de los píxeles están entre 0 y 255. Completa el código para llevarlos al rango `[0,1]`.


In [ ]:
# TODO 1 — Normalización
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print(X_train.min(), X_train.max())


In [ ]:
assert X_train.dtype == np.float32
assert X_test.dtype == np.float32
assert X_train.min() >= 0.0
assert X_train.max() <= 1.0
print("✓ Datos normalizados correctamente")


## 1.3 Visualización

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Construiremos la misma arquitectura conceptual vista en clase:

```text
Imagen 32×32
   ↓
Flatten
   ↓
1024 valores
   ↓
Dense(128)
   ↓
ReLU
   ↓
Dense(10)
   ↓
Logits
```

> La última capa **no** debe tener Softmax. Trabajaremos con logits.


In [ ]:
# TODO 2 — Completa la arquitectura
model = keras.Sequential([
    keras.Input(shape=(32, 32)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10)
])

model.summary()


In [ ]:
expected_params = (1024 * 128 + 128) + (128 * 10 + 10)
assert model.count_params() == expected_params
print(f"✓ Arquitectura correcta: {model.count_params():,} parámetros")


### Preguntas

2. ¿Por qué usamos `Flatten()` antes de la primera capa `Dense`?
3. ¿Qué pasaría si quitamos la función de activación `relu` de la capa oculta?
4. ¿Por qué la última capa tiene 10 neuronas?

**Respuestas:**

- **Pregunta 2:** `Flatten()` proyecta la matriz bidimensional $32 \times 32$ a un tensor unidimensional continuo de 1,024 elementos, formato requerido por las capas completamente conectadas `Dense`.
- **Pregunta 3:** Si se omite la activación no lineal `relu`, sucesivas transformaciones matriciales afines colapsan algebraicamente en una única transformación lineal, reduciendo la red a un modelo lineal sin capacidad para aprender fronteras no lineales.
- **Pregunta 4:** Porque CIFAR-10 posee $K = 10$ categorías disjuntas; cada neurona de salida computa el logit asociado a una clase.


---
# 3. Configurar el entrenamiento

Usaremos:

- `Adam`
- `SparseCategoricalCrossentropy`
- `accuracy`

Como la red produce logits, la pérdida debe configurarse con `from_logits=True`.


In [ ]:
lr_rate = 0.001

# TODO 3 — Configura el entrenamiento
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


### Pregunta 5

¿Por qué usamos `from_logits=True` en la función de pérdida?

**Respuesta:**

Porque la capa de salida genera logits sin activación Softmax. La formulación `from_logits=True` combina el cálculo del Softmax con la entropía cruzada mediante la formulación analítica Log-Sum-Exp, garantizando estabilidad numérica y mitigando riesgos de desbordamiento en aritmética de punto flotante.


---
# 4. Entrenar el modelo

Usaremos 10 épocas y mini-batches de 64 ejemplos. Completa `fit()`.


In [ ]:
epochs = 10
batch_size = 64

# TODO 4 — Entrenamiento
history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    shuffle=True
)


### Preguntas

6. ¿Qué representa una época?
7. Si `batch_size = 64`, ¿cuántas imágenes se usan en cada actualización de pesos?

**Respuestas:**

- **Pregunta 6:** Una época representa una pasada exhaustiva del algoritmo de optimización sobre las 50,000 imágenes de entrenamiento.
- **Pregunta 7:** Se utilizan 64 imágenes por actualización para estimar el gradiente estocástico del lote.


## 4.1 Curvas de aprendizaje

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()
plt.show()


### Pregunta 8

Observa las curvas de loss y accuracy:

- ¿Hay evidencia de overfitting? ¿Cómo lo sabes?

**Respuesta:**

Sí. A partir de la época 4, la pérdida de validación deja de descender e inicia una tendencia creciente, mientras que la pérdida de entrenamiento continúa disminuyendo. De manera análoga, el accuracy de entrenamiento alcanza $\approx 48\%$ mientras que en validación se estanca en $\approx 40\%$, confirmando memorización del conjunto de entrenamiento.


---
# 5. Evaluar el modelo

Completa `evaluate()`.


In [ ]:
# TODO 5 — Evaluación
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")


### Pregunta 9

Compara los resultados de CIFAR-10 con los que obtuvimos en clase con Fashion-MNIST:

- ¿Cuál dataset obtuvo mejor accuracy con una red similar?
- ¿Por qué crees que CIFAR-10 es más difícil para un MLP, incluso en escala de grises?

**Respuesta:**

Fashion-MNIST obtuvo un desempeño muy superior ($\approx 88\%$ frente a $\approx 40\%$ en CIFAR-10). CIFAR-10 es notablemente más complejo porque contiene objetos naturales con fondos heterogéneos, variabilidad de ángulo e iluminación y alta varianza intra-clase. El MLP aplana los píxeles perdiendo la correlación bidimensional, careciendo de la invariancia espacial local que sí aportan las capas convolucionales.


---
# 6. Logits y Softmax

La salida de la red son 10 logits. Para interpretarlos como probabilidades necesitamos Softmax.


In [ ]:
index = 0
image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Clase real: {class_names[true_class]}")
plt.axis("off")
plt.show()

image_batch = np.expand_dims(image, axis=0)
logits = model(image_batch, training=False)

print("Logits:")
print(keras.ops.convert_to_numpy(logits))


In [ ]:
# TODO 6 — Logits -> probabilidades
probabilities = keras.ops.softmax(logits)

# TODO 7 — Clase con mayor probabilidad
prediction = keras.ops.argmax(probabilities, axis=1)
prediction = int(keras.ops.convert_to_numpy(prediction)[0])

print("Clase real     :", class_names[true_class])
print("Clase predicha :", class_names[prediction])


In [ ]:
probabilities_np = keras.ops.convert_to_numpy(probabilities)[0]

print("\nProbabilidades:")
for i, p in enumerate(probabilities_np):
    marker = " ← predicción" if i == prediction else ""
    print(f"{class_names[i]:12s}: {p:.4f}{marker}")

print("\nSuma:", probabilities_np.sum())


### Pregunta 10

- ¿Qué ventajas tiene una CNN sobre una red densa para procesar imágenes?
- ¿Conservar el color (RGB, 3 canales) habría ayudado a la red? ¿Por qué?

**Respuesta:**

Las CNN introducen pesos compartidos y campos receptivos locales, preservando la topología 2D con invariancia a traslaciones y menor número de parámetros libres. Conservar los 3 canales RGB habría aportado información cromática discriminante esencial (ej. ranas verdes, cielo azul en aviones), facilitando la separación de clases difíciles.


---
# 7. Varias predicciones

Esta parte está resuelta. Observa especialmente los errores del modelo.


In [ ]:
n = 12
logits_batch = model(X_test[:n], training=False)
predictions = keras.ops.argmax(logits_batch, axis=1)
predictions = keras.ops.convert_to_numpy(predictions)

plt.figure(figsize=(12, 8))
for i in range(n):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    real = class_names[y_test[i]]
    pred = class_names[predictions[i]]
    plt.title(f"Real: {real}\nPred: {pred}", fontsize=9)
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


## 8. Diagnóstico de errores y límites arquitecturales

El accuracy final en test se sitúa en $40.0\%$ con un costo de $1.71$. Las principales confusiones ocurren entre clases biológicas morfológicamente afines (`cat` frente a `dog`, o `deer` frente a `horse`), donde la pérdida de la información espectral de color incrementa la ambigüedad de contorno.

La transformación `Flatten()` destruye por construcción la estructura topológica 2D: dos píxeles verticalmente contiguos quedan separados por 32 posiciones en memoria, impidiendo explotar correlaciones locales y contornos continuos. Esta limitación teórica fundamenta el uso de Redes Convolucionales (CNNs), cuyos filtros espaciales compartidos imponen equivariancia ante traslación y reducen drásticamente el número de parámetros.
